In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251022_021946.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/4000_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_4000_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_4000_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_4000_train_500_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id        subreddit  \
 0  1fyedxh  TwoXChromosomes   
 1  1dxp81e         abortion   
 2   8cusgg          assault   
 3  1dzgb7t         abortion   
 4   99yi99          assault   
 
                                                title  \
 0            I took the abortion pill. I’m not okay.   
 1                   2nd abortion and I feel horrible   
 2  am i allowed to feel guilty? does this even co...   
 3  Crying a few hours after abortion but I’m not ...   
 4                        My Good Friend Assaulted Me   
 
                                             selftext          created_utc  \
 0  I’m 20 nearing 21, I’ve been in a committed re...  2024-10-07 18:10:45   
 1  I feel like a scummy p.o.s.  I had a medical a...  2024-07-07 19:53:40   
 2  basically i went on a “walk” last night with a...   2018-04-17 7:35:20   
 3  I was excited to have it done and i am so reli...  2024-07-09 23:01:40   
 4  Writing this now is making me feel nauseous an...  2018-08-24 15:45:

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test12_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test12_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test12_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test12_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

In [4]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [3]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test12_emb_A, test12_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_4000_train_500_test.json
